In [1]:
import numpy as np
from typing import List
import random
from microlensing import microlensing
import pandas as pd
import matplotlib.pyplot as plt
import ms_helper as msh


In [2]:
class Param:
    def __init__(self, min, max, n= 11) -> None: #n should be odd
        self.min = min
        self.max = max
        self.n = n
    
    def range(self):
        return np.linspace(self.min,self.max,num=self.n)
    
    def get_new_range(self, value):
        width = (self.max-self.min) / (self.n-1)
        #print (self, " | " ,value, " | ",  Param(max(value - width, self.min), min(value+width, self.max)))
        return Param(max(value - width, self.min), min(value+width, self.max))
        #return Param(value - width, value+width)
    
    def __str__(self):
        return f"min: {self.min}, max: {self.max}"




In [47]:
def do_func(params, x):
    u_min = params[0]
    T0 = params[1]
    tau = params[2]
    f_bl = params[3]
    u_t =np.sqrt(u_min**2+(x-T0)**2/tau**2)
    mu = (u_t**2+2)/(u_t*np.sqrt(u_t**2+4))
    return f_bl*mu+(1-f_bl)

def find_chi(params: List[float], ypoints) -> int:
    return sum(((do_func(params, ypoints.x) - ypoints.y)/(ypoints.err))**2)


def min_chi_on_params(ypoints , params: List[Param]):
    n_params = []  # list of new parameters range 

    params_combinations = np.array(np.meshgrid(*list(map(lambda param: param.range(), params)))).T.reshape(-1,len(params))
    chis = list(map(lambda params_combination:find_chi(params_combination, ypoints), params_combinations))

    min_param_comb = params_combinations[np.argmin(chis)]
    for i in range(len(min_param_comb)):
                    n_params.append(params[i].get_new_range(min_param_comb[i]))

    return n_params, chis , params_combinations

def find_chi_repete_params(ypoints, params: List[Param], res_chi = 0.00001):
    min_chi = float('inf')
    counter = 0
    n_min_chi = 1000000000000000000000000
    all_chis = []
    all_params_comb = []
    while (min_chi - n_min_chi) > res_chi:
        min_chi = n_min_chi

        params,  chis , params_combinations = min_chi_on_params(ypoints, params)
        all_params_comb.extend(params_combinations)
        all_chis.extend(chis)
        n_min_chi = min(chis)
        min_param_comb = params_combinations[np.argmin(chis)]
        thepupik = " ".join(str(s) for s in params)
        print(f"{counter} \n old chi: {min_chi} | new chi: {n_min_chi} | min_comb: {min_param_comb}\n param: {thepupik} \n")
        
        counter +=1

    #print(f"finished!! chi: {n_min_chi} | min_comb: {min_param_comb} \n")
    return all_chis, all_params_comb 

def plot_2d_contours(chis, params_comb):
    chis=np.array(chis)
    u_min, T0 = zip(*params_comb)
    levels = np.array([0,2.30,4.61,6.17])+min(chis)
    fig, ax2 = plt.subplots()
    cs = ax2.tricontour(u_min,T0,chis, levels=levels, linewidths=0.5,colors=('red',  'green', 'orange'))
    #cntr2 = ax2.tricontourf(u_min,T0,chis, levels=levels, cmap='Blues')
    ax2.clabel(cs, inline=True, fontsize=10)

    #fig.colorbar(cntr2, ax=ax2)
    #ax2.plot(u_min,T0, 'ko', ms=1)
    ax2.set_title('u_min vs. T0 effect on Goodness of Fit (chi)')

    plt.show()

def chis_for_contours(chis, params_comb, ypoints):
    fit_params = params_comb[np.argmin(chis)] #הפרמטרים המינימלים
    df = pd.DataFrame(params_comb)
    df['chis'] = np.array(chis)
    min_chi = min(df['chis'])
    df = df[df['chis']<min_chi+60].drop(columns=['chis'])
    bounds= df.max()
    params: List[Param] = [] 
    for i in range(len(fit_params)):
        res = abs(fit_params[i]-bounds[i])
        param = Param(fit_params[i]-res,fit_params[i]+res,23)
        params.append(param)
        print(param)
    print(params)
    n_params, chis , params_combinations=min_chi_on_params(ypoints, params)
    return chis , params_combinations


def params_errors(chis, params_combinations):
    fit_params = dict(enumerate(params_combinations[np.argmin(chis)].flatten()))
    min_chi = min(chis)
    df = pd.DataFrame(params_combinations)
    df['chis'] = np.array(chis)
    #print(df[df['chis']<min_chi+1])
    df = df[df['chis']>min_chi+1]
    error_df = pd.DataFrame(columns=['value', 'max_bound', 'min_bound', 'up_error' , 'down_error'])

    for p_index in fit_params:
        const_params = fit_params.copy()
        const_params.pop(p_index)
        temp_df=df
  
        for key in const_params:    # make all other params const on their min chi value
            temp_df=temp_df[temp_df[key]==const_params[key]] 
            print(key)
            print(temp_df)
        big_df = temp_df[temp_df[p_index]>fit_params[p_index]]
        print(temp_df[p_index])
        small_df = temp_df[temp_df[p_index]<fit_params[p_index]]
        max_bound = big_df[big_df['chis']==big_df['chis'].min()].reset_index()[p_index][0]
        min_bound = small_df[small_df['chis']==small_df['chis'].min()].reset_index()[p_index][0]
        down_error = fit_params[p_index]- min_bound
        up_error = max_bound - fit_params[p_index]
        error_df.loc[p_index] = [fit_params[p_index],max_bound,min_bound,up_error,down_error]
        #       print (p_index , p )
     #       print(const_params)
    return error_df

def plot_non_linear_fit(ypoints, params):
    f_x = do_func(params , ypoints.x)
    plt.errorbar(x = ypoints.x, y = ypoints.y, yerr = ypoints.err, fmt = 'o', markersize = 0.5)
    plt.plot(ypoints.x,f_x)
    plt.xlabel('T0 (days) -2458000')
    plt.ylabel('I/I_0')
    plt.title('plot of 2D non linear fit')
    plt.grid()
    plt.show()

    plt.errorbar(x = ypoints.x, y = ypoints.y-f_x, yerr = ypoints.err, fmt='o', markersize=2)
    plt.xlabel('T0 (days) -2458000')
    plt.ylabel('yi-f(xi)')
    plt.axhline(y = 0, linestyle = '--')
    plt.title('residuals plot for 2D non linear fit')
    plt.grid()
    plt.show()

def print_nsigma(ogle_name,fit_val):
        print("")
        print(f"nsigma with ogle {ogle_name} parameter")
        print(ms_event.ogle[ogle_name])
        print(fit_val)
        print(f"nsigma: {msh.nsigma(ms_event.ogle[ogle_name], fit_val)}")
        print("\n")


In [4]:
ms_event = microlensing("https://www.astrouw.edu.pl/ogle/ogle4/ews/2019/blg-0035")
ms_event.data['norm_time'] =ms_event.data['JHD']-2458000

In [5]:
params: List[Param] = [Param(0.676,0.682,11), #umin
                       Param(590,590.6,11),   #T0
                       Param(60.6,61,11),     #tau
                       Param(0.8,1.1,11)]     #f_bl
ypoints= ms_event.data[['norm_time','I','I_error']].set_axis(['x','y','err'],axis=1)

In [ ]:
all_chis, all_params_comb  = find_chi_repete_params(ypoints, params)

In [ ]:
min_chi_params = all_params_comb[np.argmin(all_chis)]
plot_non_linear_fit(ypoints, min_chi_params)

In [ ]:
chis , params_combinations = chis_for_contours(all_chis, all_params_comb, ypoints)

In [ ]:
#plot_2d_contours(chis, params_combinations)

In [ ]:
df = pd.DataFrame(params_combinations)
df['chis'] = np.array(chis)
min_chi=min(chis)
print(min_chi)
fit_params = params_combinations[np.argmin(chis)]
#df = df[df['chis']>min_chi+1]
df[(df[1]==fit_params[1])&(df[2]==fit_params[2])&(df[3]==fit_params[3])]

In [ ]:
print(params_combinations[np.argmin(chis)])
print(len(params_combinations))
error_df = params_errors(chis , params_combinations)
print(error_df)

In [ ]:
u_min_fit = msh.value_with_error('u_min fit',error_df.loc[0].value , error_df.loc[0].up_error)
T0_fit = msh.value_with_error('T0 fit',error_df.loc[1].value , error_df.loc[1].up_error)

In [ ]:
print_nsigma('umin',u_min_fit)
T0_fit.value = T0_fit.value+2458000
print_nsigma('t0',T0_fit)